# LexicoTrend FR — Analyse exploratoire

> **Richesse lexicale des romans best-sellers français (1850–1980)**

Ce notebook documente la démarche analytique complète :
- Exploration et validation du corpus
- Distribution des métriques lexicales
- Test des 3 hypothèses de recherche
- Visualisation des résultats ML

**Prérequis :** avoir exécuté l'intégralité de la pipeline :
```bash
python init_db.py
python scraping/gallica.py
python scraping/gutenberg.py
python processing/clean.py --batch
python processing/ocr_corrector.py --batch
python processing/metrics.py --batch
python ml/analysis.py
```

## 0. Setup

In [ ]:
import sys
import json
import sqlite3
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats as scipy_stats

warnings.filterwarnings('ignore')

# Ajouter la racine du projet au path
sys.path.insert(0, str(Path('..').resolve()))
from init_db import get_connection

# Style matplotlib
plt.style.use('dark_background')
PALETTE = {
    'primary': '#1E3A5F',
    'accent':  '#E8853D',
    'success': '#2E7D5E',
    'warning': '#C0392B',
    'neutral': '#6B7280',
}

DB_PATH       = '../data/lexicotrend.db'
ANALYSIS_JSON = Path('../data/processed/analysis_results.json')

print('Setup OK ✓')

## 1. Chargement des données

In [ ]:
conn = get_connection(DB_PATH)

df = pd.read_sql_query("""
    SELECT
        book_id, title, author, year, decade, genre, source,
        word_count, unique_words, sentence_count,
        ttr, mtld, hdd, mtld_ma_bid,
        ocr_quality_before, ocr_quality_after, ocr_corrected,
        is_outlier, claude_interpretation
    FROM books
    ORDER BY year
""", conn)

conn.close()

df['genre']        = df['genre'].fillna('inconnu')
df['decade_label'] = df['decade'].astype(str) + 's'

print(f'Corpus total : {len(df)} livres')
print(f'Avec MTLD   : {df["mtld"].notna().sum()} livres')
print(f'Décennies   : {df["decade"].nunique()} ({df["decade"].min()}–{df["decade"].max()})')
print(f'Genres      : {df["genre"].unique().tolist()}')
print(f'Sources     : {df["source"].value_counts().to_dict()}')

In [ ]:
# Vue d'ensemble du DataFrame
df[df['mtld'].notna()].describe().round(2)

## 2. Qualité du corpus

In [ ]:
# Distribution de la couverture temporelle
decade_counts = df.groupby('decade_label').size().reset_index(name='n_books')

fig = px.bar(
    decade_counts,
    x='decade_label', y='n_books',
    title='Nombre de livres par décennie',
    color='n_books',
    color_continuous_scale='Blues',
    labels={'decade_label': 'Décennie', 'n_books': 'Nombre de livres'}
)
fig.update_layout(showlegend=False, height=350)
fig.show()

print('\nDécennies sous-représentées (< 3 livres) :')
print(decade_counts[decade_counts['n_books'] < 3])

In [ ]:
# Qualité OCR avant / après correction
df_ocr = df[df['ocr_quality_before'].notna()].copy()

if not df_ocr.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(df_ocr['ocr_quality_before'], bins=20, color=PALETTE['warning'], alpha=0.8, edgecolor='white')
    axes[0].axvline(df_ocr['ocr_quality_before'].median(), color=PALETTE['accent'], linewidth=2, label=f'Médiane : {df_ocr["ocr_quality_before"].median():.3f}')
    axes[0].set_title('Score OCR avant correction', color='white')
    axes[0].set_xlabel('Score qualité (0–1)')
    axes[0].legend()

    df_after = df[df['ocr_quality_after'].notna()]
    if not df_after.empty:
        axes[1].hist(df_after['ocr_quality_after'], bins=20, color=PALETTE['success'], alpha=0.8, edgecolor='white')
        axes[1].axvline(df_after['ocr_quality_after'].median(), color=PALETTE['accent'], linewidth=2, label=f'Médiane : {df_after["ocr_quality_after"].median():.3f}')
        axes[1].set_title('Score OCR après correction', color='white')
        axes[1].set_xlabel('Score qualité (0–1)')
        axes[1].legend()

    plt.tight_layout()
    plt.savefig('../data/processed/figures/ocr_quality_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Livres corrigés par ByT5 : {df["ocr_corrected"].sum()}')
else:
    print('Scores OCR non disponibles — exécuter la pipeline de collecte d abord')

## 3. Distribution des métriques lexicales

In [ ]:
df_m = df[df['mtld'].notna()].copy()

if df_m.empty:
    print('Aucune métrique disponible. Exécuter processing/metrics.py --batch')
else:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    metrics = [
        ('mtld',        'Score MTLD',          PALETTE['accent']),
        ('ttr',         'Type-Token Ratio',     PALETTE['primary']),
        ('hdd',         'Score HD-D',           PALETTE['success']),
        ('word_count',  'Nombre de tokens',     PALETTE['neutral']),
    ]

    for ax, (col, label, color) in zip(axes.flatten(), metrics):
        data = df_m[col].dropna()
        if data.empty:
            ax.set_title(f'{label} — non disponible', color='white')
            continue
        ax.hist(data, bins=25, color=color, alpha=0.8, edgecolor='#0F1B2D')
        ax.axvline(data.median(), color=PALETTE['accent'], linewidth=2,
                   label=f'Médiane : {data.median():.2f}')
        ax.axvline(data.mean(), color='white', linewidth=1.5, linestyle='--',
                   label=f'Moyenne : {data.mean():.2f}')
        ax.set_title(label, color='white', fontsize=12)
        ax.set_xlabel(col)
        ax.legend(fontsize=9)

    plt.suptitle('Distribution des métriques lexicales', fontsize=14, color='white', y=1.02)
    plt.tight_layout()
    Path('../data/processed/figures').mkdir(parents=True, exist_ok=True)
    plt.savefig('../data/processed/figures/metrics_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Corrélation entre métriques
if not df_m.empty:
    corr_cols = ['mtld', 'ttr', 'hdd', 'word_count', 'year']
    corr_df   = df_m[corr_cols].dropna()

    if not corr_df.empty:
        corr_matrix = corr_df.corr()
        fig = px.imshow(
            corr_matrix,
            color_continuous_scale='RdBu_r',
            zmin=-1, zmax=1,
            title='Matrice de corrélation des métriques',
            text_auto='.2f'
        )
        fig.update_layout(height=450)
        fig.show()

        # Point clé : si MTLD et TTR sont très corrélés, MTLD apporte moins d'information
        mtld_ttr_corr = corr_matrix.loc['mtld', 'ttr']
        print(f'Corrélation MTLD ↔ TTR : {mtld_ttr_corr:.3f}')
        print('→ MTLD est indépendant de la longueur du texte si corrélation faible avec word_count')
        print(f'Corrélation MTLD ↔ word_count : {corr_matrix.loc["mtld", "word_count"]:.3f}')

## 4. Évolution temporelle — Hypothèse H1

In [ ]:
if not df_m.empty:
    decade_stats = df_m.groupby(['decade', 'decade_label'])['mtld'].agg(
        mean='mean', median='median', std='std', count='count'
    ).reset_index().sort_values('decade')

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Médiane MTLD par décennie', 'Boxplots MTLD par décennie'),
        column_widths=[0.4, 0.6]
    )

    # Courbe médiane
    fig.add_trace(go.Scatter(
        x=decade_stats['decade_label'],
        y=decade_stats['median'],
        mode='lines+markers',
        name='Médiane MTLD',
        line=dict(color=PALETTE['accent'], width=2.5),
        marker=dict(size=9, color=PALETTE['accent'])
    ), row=1, col=1)

    # Intervalle ±1 std
    fig.add_trace(go.Scatter(
        x=list(decade_stats['decade_label']) + list(decade_stats['decade_label'])[::-1],
        y=list(decade_stats['median'] + decade_stats['std'].fillna(0)) +
          list((decade_stats['median'] - decade_stats['std'].fillna(0)))[::-1],
        fill='toself',
        fillcolor=f"{PALETTE['accent']}22",
        line=dict(color='rgba(0,0,0,0)'),
        name='±1 écart-type'
    ), row=1, col=1)

    # Boxplots
    for decade in sorted(df_m['decade'].unique()):
        subset = df_m[df_m['decade'] == decade]
        fig.add_trace(go.Box(
            y=subset['mtld'],
            name=str(decade) + 's',
            marker_color=PALETTE['primary'],
            line_color=PALETTE['accent'],
            fillcolor=f"{PALETTE['primary']}55",
            showlegend=False
        ), row=1, col=2)

    fig.update_layout(
        title='H1 — Évolution du MTLD dans le temps',
        height=500,
        plot_bgcolor='#0F1B2D',
        paper_bgcolor='#0F1B2D',
        font=dict(color='#E8E0D5')
    )
    fig.show()
    fig.write_image('../data/processed/figures/h1_temporal_evolution.png', scale=2)

In [ ]:
# Test statistique H1 : tendance temporelle (OLS)
if not df_m.empty:
    import statsmodels.api as sm

    X = sm.add_constant(df_m['year'].values)
    y = df_m['mtld'].values
    model = sm.OLS(y, X).fit()

    print('═' * 50)
    print('RÉGRESSION OLS : année → MTLD')
    print('═' * 50)
    print(model.summary())

    print('\n─── Résumé H1 ───')
    slope  = model.params[1]
    pvalue = model.pvalues[1]
    r2     = model.rsquared
    print(f'Pente    : {slope:+.6f} (MTLD/an)')
    print(f'p-value  : {pvalue:.6f}')
    print(f'R²       : {r2:.6f}')

    if pvalue < 0.05:
        direction = 'diminue' if slope < 0 else 'augmente'
        print(f'\n✅ H1 SUPPORTÉE — MTLD {direction} de {abs(slope)*10:.3f} points par décennie (significatif)')
    else:
        print(f'\n❌ H1 INFIRMÉE — pas de tendance significative (p={pvalue:.4f})')

    # Test avant/après 1945
    pre  = df_m[df_m['year'] <= 1945]['mtld']
    post = df_m[df_m['year'] >  1945]['mtld']
    if len(pre) >= 3 and len(post) >= 3:
        t, p = scipy_stats.ttest_ind(pre, post, equal_var=False)
        print(f'\nTest Welch avant/après 1945 :')
        print(f'  Avant 1945 : M = {pre.mean():.2f} (n={len(pre)})')
        print(f'  Après 1945 : M = {post.mean():.2f} (n={len(post)})')
        print(f'  t = {t:.4f}, p = {p:.6f}')

## 5. Variance intra-décennie — Hypothèse H2

In [ ]:
if not df_m.empty:
    decade_var = df_m.groupby('decade')['mtld'].std().reset_index()
    decade_var.columns = ['decade', 'mtld_std']
    decade_var['period'] = decade_var['decade'].apply(
        lambda d: 'Avant 1920' if d < 1920 else 'Après 1920'
    )

    fig = px.bar(
        decade_var,
        x='decade', y='mtld_std',
        color='period',
        title='H2 — Variance intra-décennie du MTLD',
        labels={'decade': 'Décennie', 'mtld_std': 'Écart-type MTLD', 'period': 'Période'},
        color_discrete_map={'Avant 1920': PALETTE['accent'], 'Après 1920': PALETTE['primary']}
    )
    fig.add_hline(
        y=df_m[df_m['decade'] < 1920]['mtld'].std(),
        line_dash='dot', line_color=PALETTE['accent'],
        annotation_text='Écart-type avant 1920'
    )
    fig.update_layout(height=400, plot_bgcolor='#0F1B2D', paper_bgcolor='#0F1B2D',
                      font=dict(color='#E8E0D5'))
    fig.show()

    # Test de Levene
    pre_1920  = df_m[df_m['decade'] <  1920]['mtld'].dropna()
    post_1920 = df_m[df_m['decade'] >= 1920]['mtld'].dropna()

    if len(pre_1920) >= 3 and len(post_1920) >= 3:
        levene_stat, levene_p = scipy_stats.levene(pre_1920, post_1920)
        print('─── Test de Levene (égalité des variances) ───')
        print(f'Avant 1920 : σ = {pre_1920.std():.3f} (n={len(pre_1920)})')
        print(f'Après 1920 : σ = {post_1920.std():.3f} (n={len(post_1920)})')
        print(f'Levene F = {levene_stat:.4f}, p = {levene_p:.6f}')

        if levene_p < 0.05 and pre_1920.std() > post_1920.std():
            print('\n✅ H2 SUPPORTÉE — variance significativement plus élevée avant 1920')
        elif levene_p < 0.05:
            print('\n⚠️ H2 INFIRMÉE — différence significative mais dans la direction inverse')
        else:
            print(f'\n❌ H2 INFIRMÉE — pas de différence significative (p={levene_p:.4f})')

## 6. Genre vs Décennie — Hypothèse H3

In [ ]:
if not df_m.empty:
    # MTLD par genre
    genre_stats = df_m.groupby('genre')['mtld'].agg(
        median='median', std='std', count='count'
    ).reset_index().sort_values('median', ascending=False)

    fig = px.box(
        df_m, x='genre', y='mtld',
        title='H3 — Distribution MTLD par genre littéraire',
        color='genre',
        labels={'genre': 'Genre', 'mtld': 'Score MTLD'},
        color_discrete_sequence=px.colors.qualitative.Set2
    )
    fig.update_layout(showlegend=False, height=400,
                      plot_bgcolor='#0F1B2D', paper_bgcolor='#0F1B2D',
                      font=dict(color='#E8E0D5'))
    fig.show()

    print('Médiane MTLD par genre :')
    print(genre_stats.to_string(index=False))

In [ ]:
# Feature importance Random Forest (chargée depuis analysis_results.json)
if ANALYSIS_JSON.exists():
    with open(ANALYSIS_JSON) as f:
        analysis = json.load(f)

    rf = analysis.get('random_forest_h3', {})
    fi = rf.get('feature_importances', {})

    if fi:
        fi_df = pd.DataFrame({
            'Feature':    list(fi.keys()),
            'Importance': list(fi.values())
        }).sort_values('Importance', ascending=True)

        fig = go.Figure(go.Bar(
            x=fi_df['Importance'],
            y=fi_df['Feature'],
            orientation='h',
            marker_color=[
                PALETTE['accent'] if f in ['genre', 'decade'] else PALETTE['primary']
                for f in fi_df['Feature']
            ],
            text=[f'{v:.4f}' for v in fi_df['Importance']],
            textposition='outside'
        ))
        fig.update_layout(
            title='H3 — Feature importance Random Forest (prédiction MTLD)',
            xaxis_title='Importance (MDI)',
            plot_bgcolor='#0F1B2D', paper_bgcolor='#0F1B2D',
            font=dict(color='#E8E0D5'),
            height=350
        )
        fig.show()

        h3 = rf.get('h3_genre_vs_decade', {})
        print(f"\nGenre importance   : {h3.get('genre_importance', 'N/A')}")
        print(f"Décennie importance: {h3.get('decade_importance', 'N/A')}")
        status = '✅ H3 SUPPORTÉE' if h3.get('h3_supported') else '❌ H3 INFIRMÉE'
        print(f"\n{status}")
        print(f"OOB R² : {rf.get('oob_r_squared', 'N/A')}")
else:
    print('analysis_results.json absent. Exécuter : python ml/analysis.py')

## 7. Clustering KMeans

In [ ]:
if ANALYSIS_JSON.exists():
    km = analysis.get('kmeans_clustering', {})
    book_clusters = km.get('book_clusters', [])

    if book_clusters:
        df_clusters = pd.DataFrame(book_clusters)
        df_clusters['cluster'] = df_clusters['cluster'].astype(str)

        fig = px.scatter(
            df_clusters,
            x='decade', y='mtld',
            color='cluster',
            symbol='genre',
            hover_data=['book_id', 'genre'],
            title=f'Clustering KMeans — k={km.get("k_optimal","?")} optimal '
                  f'(silhouette={km.get("best_silhouette",0):.4f})',
            labels={'decade': 'Décennie', 'mtld': 'Score MTLD', 'cluster': 'Cluster'},
            color_discrete_sequence=px.colors.qualitative.Bold
        )
        fig.update_layout(
            height=500,
            plot_bgcolor='#0F1B2D', paper_bgcolor='#0F1B2D',
            font=dict(color='#E8E0D5')
        )
        fig.show()

        print('Scores silhouette par k :')
        for k, score in sorted(km.get('silhouette_scores', {}).items()):
            marker = '← optimal' if int(k) == km.get('k_optimal') else ''
            print(f'  k={k} : {score:.4f} {marker}')
else:
    print('analysis_results.json absent. Exécuter : python ml/analysis.py')

## 8. Outliers et interprétations Claude

In [ ]:
outliers = df[df['is_outlier'] == 1].sort_values('mtld', ascending=False)

print(f'{len(outliers)} outliers détectés\n')

if not outliers.empty:
    for _, row in outliers.iterrows():
        print(f"{'─'*60}")
        print(f"📖 {row['title']} — {row['author']} ({row['year']})")
        print(f"   Genre : {row['genre']} | Source : {row['source']}")
        print(f"   MTLD  : {row['mtld']:.2f}")
        if row.get('claude_interpretation'):
            print(f"   💬 {row['claude_interpretation']}")
        else:
            print("   ⏳ Interprétation non encore générée")
    print(f"{'─'*60}")

## 9. Résumé des hypothèses

In [ ]:
if ANALYSIS_JSON.exists():
    hyp = analysis.get('hypotheses_summary', {})

    print('═' * 60)
    print('RÉSUMÉ DES HYPOTHÈSES DE RECHERCHE')
    print('═' * 60)

    for h_name, h_data in hyp.items():
        supported = h_data.get('supported')
        icon = '✅' if supported else ('❌' if supported is False else '⚠️')
        status = 'SUPPORTÉE' if supported else ('INFIRMÉE' if supported is False else 'INDÉTERMINÉE')
        print(f'\n{icon} {h_name.replace("_", " ")} : {status}')
        for k, v in h_data.items():
            if k != 'supported' and v is not None:
                print(f'   {k}: {v}')

    print('\n' + '═' * 60)
    print(f"Livres analysés : {analysis.get('n_books_analyzed', 'N/A')}")
    print(f"Outliers        : {analysis.get('n_outliers', 'N/A')}")
    print(f"Généré le       : {analysis.get('generated_at', 'N/A')}")
else:
    print('Lancer python ml/analysis.py pour générer les résultats')

## 10. Export des figures pour le README

In [ ]:
from pathlib import Path
figures_dir = Path('../data/processed/figures')
figures_dir.mkdir(parents=True, exist_ok=True)

figures = list(figures_dir.glob('*.png'))
print(f'{len(figures)} figures exportées dans {figures_dir} :')
for f in sorted(figures):
    size_kb = f.stat().st_size // 1024
    print(f'  {f.name} ({size_kb} Ko)')

print('\n→ Copier les figures pertinentes dans docs/ pour le README GitHub')